# DPO Training with Unsloth

Fine-tune a judge model using Direct Preference Optimization.


In [ ]:
import os
import torch
from accelerate import Accelerator
from datasets import load_from_disk

from dpo_training_helpers import (
    preprocess_dpo,
    load_unsloth_model,
    apply_lora,
    build_dpo_trainer,
)

In [ ]:
accelerator = Accelerator()
print("Using accelerator:", accelerator.state)
print("Num processes:", accelerator.num_processes)

In [ ]:
import os
from pathlib import Path
import torch

MAX_SEQ_LENGTH = 2048
MODEL_NAME = "Qwen/Qwen2-7B-Instruct"

REPO_ROOT = Path(__file__).resolve().parents[1]


TRAIN_PATH = REPO_ROOT / "Skydpo_dataset" / "train"
VAL_PATH   = REPO_ROOT / "Skydpo_dataset" / "validation"

OUTPUT_DIR = REPO_ROOT / "models" / "SkyDPO_Qwen27"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


In [ ]:
train_dataset = load_from_disk(TRAIN_PATH)
eval_dataset  = load_from_disk(VAL_PATH)

print("Train columns:", train_dataset.column_names)
print("Eval columns:", eval_dataset.column_names)


In [ ]:
train_dataset = train_dataset.map(
    preprocess_dpo,
    remove_columns=train_dataset.column_names,
)

eval_dataset = eval_dataset.map(
    preprocess_dpo,
    remove_columns=eval_dataset.column_names,
)

print("Train size:", len(train_dataset))
print("Eval size:", len(eval_dataset))
print("Sample:", train_dataset[0])


In [ ]:
model, tokenizer = load_unsloth_model(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
)

model = apply_lora(model)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))

In [ ]:
trainer = build_dpo_trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    output_dir=OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
)

In [ ]:
print("Starting DPO training on 2× L4 GPUs...")
trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Training complete! Model saved to: {OUTPUT_DIR}")